# SI — the vectorial PSF is one shape, rescaled with wavelength

**Why:** we want every simulation to use the vectorial PSF of a freely rotating dipole
(`DipolePSF`; Fazel et al. 2024, Rev. Mod. Phys. 96, 025003), not the Gaussian approximation. The
full 2D calculation costs ~26 ms per wavelength and emitter position. That is too slow for
simulations where each photon has its own wavelength and each emitter its own position.

**The point of this notebook:** for an emitter in focus at the coverslip, the back-focal-plane field
depends only on the emission angle: the Fresnel coefficients, the aplanatic factor and the dipole
projections are all functions of angle, and the refractive indices are non-dispersive. The image
is the Fourier transform of that field at $k\,\mathbf{x}$, with $k = 2\pi\,\mathrm{NA}/\lambda$, so

$$I_\lambda(r) = \left(\frac{\lambda_0}{\lambda}\right)^2 I_{\lambda_0}\!\left(r\,\frac{\lambda_0}{\lambda}\right),$$

and the isotropic emitter's image is circularly symmetric. The PSF can therefore be computed **once**,
as a radial profile at $\lambda_0$, and reused for every photon and every emitter position
(`pyS3M.simulation.vectorial_sampler.RadialDipolePSF`).

1. The PSF at several wavelengths, and its collapse onto one shape when rescaled.
2. Circular symmetry.
3. The radial table against the full 2D model, pixel-integrated, across wavelengths and positions.
4. Speed.
5. Photon by photon: an emitter moving across the Bayer pattern, each photon with its own
   wavelength, detected by the pixel type it lands on, plus background and the camera.

Nothing here is fitted. The checks in §2–3 are repeated in `unit_tests/test_vectorial_sampler.py`.

---
## 0 — Setup

In [ ]:
import time
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt

from pyS3M import IOFunctions, SpectralFunctions, MaskFunctions
from pyS3M.simulation.multicolour import MultiC_Sim_Funcs
from pyS3M.simulation.defocus_psf import DipolePSF
from pyS3M.simulation.vectorial_sampler import (
    RadialDipolePSF, photon_by_photon_photoelectrons, background_photoelectrons, camera_adu)

IO = IOFunctions.IO_Functions()
S_F = SpectralFunctions.Spectral_Funcs()
M_F = MaskFunctions.Mask_Functions()
MSF = MultiC_Sim_Funcs()

REPO_ROOT = Path.cwd().resolve().parents[2]   # notebooks/figures/SI -> repo root
CALIBRATION_DIR = REPO_ROOT / "Camera_Calibrations" / "Ximea_Camera"
FIG_DIR = Path("/media/u2473660/JSBLeeLabBackup/Data/20261029_VectorialScaling")
FIG_DIR.mkdir(parents=True, exist_ok=True)

NA, N_MEDIUM, N_IMMERSION = 1.49, 1.33, 1.515   # oil objective, dipole in water at the coverslip
PIXEL_NM = 69.0                                  # Ximea
LAMBDA0_UM = 0.6                                 # reference wavelength of the radial profile
RNG_SEED = 20261029

dipole = DipolePSF(NA=NA, n_medium=N_MEDIUM, n_immersion=N_IMMERSION, N_pupil=512)
t0 = time.time()
psf = RadialDipolePSF(NA=NA, n_medium=N_MEDIUM, n_immersion=N_IMMERSION, N_pupil=512,
                      ref_wavelength_um=LAMBDA0_UM)
print(f"radial profile built in {time.time() - t0:.2f} s; {psf.enclosed:.4f} of the light within "
      f"{psf.r_um[-1]:g} um at {LAMBDA0_UM * 1000:g} nm")

---
## 1 — One shape, rescaled

Left: the full 2D PSF (`DipolePSF.image`) at three wavelengths. Middle: their radial profiles.
Right: the same profiles against $r\,\lambda_0/\lambda$, multiplied by $(\lambda/\lambda_0)^2$: they fall
on one curve. The printout gives the largest difference between the directly computed 2D image and
the rescaled $\lambda_0$ image, relative to the peak.

In [ ]:
WAVELENGTHS_UM = [0.45, 0.6, 0.75]
COLOURS = ["tab:blue", "tab:green", "tab:red"]
x = np.linspace(-0.8, 0.8, 161)   # um
r = np.linspace(0, 1.2, 241)

fig, axs = plt.subplots(1, 5, figsize=(14, 2.8), gridspec_kw=dict(width_ratios=[1, 1, 1, 1.4, 1.4]))
for ax, lam in zip(axs[:3], WAVELENGTHS_UM):
    ax.imshow(dipole.image(lam, x, x), extent=[x[0], x[-1], x[0], x[-1]], cmap="magma")
    ax.set_title(f"{lam * 1000:.0f} nm")
    ax.set_xlabel("x / um")
axs[0].set_ylabel("y / um")
for lam, c in zip(WAVELENGTHS_UM, COLOURS):
    profile = dipole.image(lam, r, np.zeros(1))[0]
    axs[3].plot(r, profile, color=c, label=f"{lam * 1000:.0f} nm")
    axs[4].plot(r * LAMBDA0_UM / lam, profile * (lam / LAMBDA0_UM) ** 2, color=c, lw=3 - WAVELENGTHS_UM.index(lam))
axs[3].set_xlabel("r / um"); axs[3].set_ylabel("intensity / um$^{-2}$"); axs[3].legend(fontsize=7)
axs[4].set_xlabel(r"r $\lambda_0/\lambda$ / um"); axs[4].set_ylabel(r"intensity $\times(\lambda/\lambda_0)^2$")
axs[4].set_xlim(0, 1.2 * LAMBDA0_UM / max(WAVELENGTHS_UM))
for ax in axs[3:]:
    ax.set_yscale("log"); ax.set_ylim(1e-3, None)
fig.tight_layout()
fig.savefig(FIG_DIR / "psf_scaling.svg", bbox_inches="tight")
plt.show()

for lam in np.arange(0.45, 0.76, 0.05):
    direct = dipole.image(lam, x, x)
    scaled = dipole.image(LAMBDA0_UM, x * LAMBDA0_UM / lam, x * LAMBDA0_UM / lam) * (LAMBDA0_UM / lam) ** 2
    print(f"{lam * 1000:.0f} nm: max |rescaled - direct| / peak = {np.abs(scaled - direct).max() / direct.max():.1e}")

---
## 2 — Circular symmetry

The isotropic emitter's image along rays at several angles, minus the image along +x, relative to
the peak. Any difference comes from the square pupil grid.

In [ ]:
r = np.linspace(0, 1.5, 151)
on_axis = dipole.image(LAMBDA0_UM, r, np.zeros(1))[0]
fig, ax = plt.subplots(figsize=(4, 2.8))
for angle in np.deg2rad([15, 30, 45, 60, 75]):
    ray = np.array([dipole.image(LAMBDA0_UM, np.array([ri * np.cos(angle)]), np.array([ri * np.sin(angle)]))[0, 0]
                    for ri in r])
    dev = (ray - on_axis) / on_axis.max()
    ax.plot(r, dev, label=f"{np.rad2deg(angle):.0f} deg")
    print(f"{np.rad2deg(angle):4.0f} deg: max |I(r, angle) - I(r, 0)| / peak = {np.abs(dev).max():.1e}")
ax.set_xlabel("r / um"); ax.set_ylabel("difference / peak"); ax.legend(fontsize=7)
fig.tight_layout()
plt.show()

---
## 3 — The radial table against the full 2D model

What a camera pixel collects: the PSF integrated over each 69 nm pixel (5 × 5 sub-samples), from
the full 2D model (`DipolePSF.pixel_patches`) and from the rescaled radial table
(`RadialDipolePSF.pixel_patch`), over the visible range and across the Bayer unit cell (offsets in
pixels from the central pixel's centre). The map shows one difference.

In [ ]:
PATCH_PX, OVERSAMPLE = 21, 5
offsets_px = [(0.0, 0.0), (0.5, 0.0), (0.5, 0.5), (1.0, 0.25), (1.5, 1.5)]
worst = 0.0
for lam in np.arange(0.45, 0.76, 0.05):
    for ox, oy in offsets_px:
        args = (PIXEL_NM / 1000, PATCH_PX, OVERSAMPLE, ox * PIXEL_NM / 1000, oy * PIXEL_NM / 1000)
        full = dipole.pixel_patches(lam, *args)[0]
        radial = psf.pixel_patch(lam, *args)
        worst = max(worst, np.abs(radial - full).max() / full.max())
print(f"largest |radial - full 2D| over 7 wavelengths x {len(offsets_px)} offsets: {worst:.1e} of the peak pixel")

args = (PIXEL_NM / 1000, PATCH_PX, OVERSAMPLE, 0.5 * PIXEL_NM / 1000, 0.25 * PIXEL_NM / 1000)
full, radial = dipole.pixel_patches(0.6, *args)[0], psf.pixel_patch(0.6, *args)
fig, axs = plt.subplots(1, 3, figsize=(9, 2.8))
for ax, img, title in zip(axs, [full, radial, (radial - full) / full.max()],
                          ["full 2D", "radial table", "difference / peak"]):
    im = ax.imshow(img, cmap="magma" if title != "difference / peak" else "RdBu_r")
    ax.set_title(title); fig.colorbar(im, ax=ax, shrink=0.8)
fig.tight_layout()
plt.show()

---
## 4 — Speed

The full 2D model needs one patch per wavelength bin and emitter position; the radial table needs
nothing per position or wavelength, only a draw per photon. Timed for 15 wavelength bins (as model C
of the Factorisation notebook) and for 1,000 frames of 1,000 photons at random positions.

In [ ]:
rng = np.random.default_rng(RNG_SEED)
lam_bins = np.linspace(0.5, 0.7, 15)
t0 = time.time()
dipole.pixel_patches(lam_bins, PIXEL_NM / 1000, PATCH_PX, OVERSAMPLE, 0.01, 0.02)
t_full = time.time() - t0

H = W = 14
pixel_type = np.zeros((H, W), dtype=int)
wl = np.arange(400.0, 801.0)
flat = np.ones_like(wl)
qe_one = np.ones((1, len(wl)))
n_frames = 1000
t0 = time.time()
pe = photon_by_photon_photoelectrons(psf, flat, wl, qe_one, pixel_type, 1000,
                                     rng.uniform(6, 8, n_frames), rng.uniform(6, 8, n_frames), PIXEL_NM, rng)
t_radial = time.time() - t0
print(f"full 2D: {t_full:.2f} s for ONE emitter position (15 wavelength bins)")
print(f"radial:  {t_radial:.2f} s for {n_frames} frames at random positions, every photon its own wavelength "
      f"({pe.sum() / t_radial / 1e6:.1f} M detected photons/s)")
print(f"-> {t_full * n_frames / t_radial:.0f}x faster for this case")

---
## 5 — Photon by photon across the Bayer pattern

Each frame: Poisson($N$) photons from the dye, each with its own wavelength drawn from the emission
spectrum, landing where the PSF at that wavelength puts it, and detected by the pixel it lands on
with that pixel type's QE at that wavelength. Background as in production (flat spectrum, each
pixel's background QE), then the Ximea camera model (calibration medians: gain, offset, read-noise
variance) gives ADU.

The emitter moves across one 2 × 2 Bayer unit cell. Rows: positions. Columns: the expected
photoelectrons (mean of many frames, noise free) and one noisy ADU frame. The last plot shows the
fraction of the detected signal on each pixel type against position.

In [ ]:
DYE = "ATTO 565"
N_PHOTONS = 2000              # collected photons per frame
BACKGROUND_PHOTONS = 5.0      # per pixel, before detection
N_EXPECTED = 2000             # frames averaged for the noise-free image

R_qe, G_qe, B_qe, wavelength = S_F.getpixelefficiency()
pixel_order = ["B", "G", "R"]
qe_curves = np.vstack([B_qe, G_qe, R_qe])
spectrum = S_F.get_dye_or_filter_data(names=[DYE], wavelength=wavelength, dye_or_filter=True)[0]

H = W = 14
masks = M_F.get_masks(size_x=H, size_y=W)
pixel_type = np.full((H, W), -1)
for i, c in enumerate(pixel_order):
    pixel_type[masks[c]] = i
bg_qe_per_type = MSF.background_pixel_efficiency(wavelength, qe_curves, [1.0, 1.0, 1.0])
bg_qe = bg_qe_per_type[pixel_type]

gain = np.median(IO.read_tiff(CALIBRATION_DIR / "gain.tif"))
offset = np.median(IO.read_tiff(CALIBRATION_DIR / "offset.tif"))
variance = np.median(IO.read_tiff(CALIBRATION_DIR / "variance.tif"))
cam = dict(gain=np.full((H, W), gain), offset=np.full((H, W), offset), variance=np.full((H, W), variance))

# a loop around one 2 x 2 unit cell (px; pixel centres at integers)
path = [(6.0, 6.0), (6.5, 6.0), (7.0, 6.0), (7.0, 6.5), (7.0, 7.0), (6.5, 7.0), (6.0, 7.0), (6.0, 6.5)]

rng = np.random.default_rng(RNG_SEED)
expected, frames, fractions = [], [], []
for x0, y0 in path:
    sig = photon_by_photon_photoelectrons(psf, spectrum, wavelength, qe_curves, pixel_type, N_PHOTONS,
                                          np.full(N_EXPECTED, x0), np.full(N_EXPECTED, y0), PIXEL_NM, rng)
    mean_sig = sig.mean(axis=0)
    expected.append(mean_sig + BACKGROUND_PHOTONS * bg_qe)
    fractions.append([mean_sig[pixel_type == i].sum() / mean_sig.sum() for i in range(3)])
    one = sig[:1] + background_photoelectrons(BACKGROUND_PHOTONS, bg_qe, 1, rng)
    frames.append(camera_adu(one, rng=rng, **cam)[0])

fig, axs = plt.subplots(2, len(path), figsize=(2 * len(path), 4.2))
for j, (x0, y0) in enumerate(path):
    axs[0, j].imshow(expected[j], cmap="magma"); axs[0, j].plot(x0, y0, "c+", ms=8)
    axs[1, j].imshow(frames[j], cmap="gray"); axs[1, j].plot(x0, y0, "c+", ms=8)
    axs[0, j].set_title(f"({x0:g}, {y0:g}) px", fontsize=8)
    for ax in axs[:, j]:
        ax.set_xticks([]); ax.set_yticks([])
axs[0, 0].set_ylabel("expected pe"); axs[1, 0].set_ylabel("one frame, ADU")
fig.tight_layout()
fig.savefig(FIG_DIR / f"bayer_path_{DYE.replace(' ', '')}.svg", bbox_inches="tight")
plt.show()

fractions = np.array(fractions)
fig, ax = plt.subplots(figsize=(4.5, 2.6))
for i, (c, col) in enumerate(zip(pixel_order, ["tab:blue", "tab:green", "tab:red"])):
    ax.plot(range(len(path)), fractions[:, i], "o-", color=col, label=f"pixel type {c}")
ax.set_xticks(range(len(path)), [f"({x:g},{y:g})" for x, y in path], rotation=45, fontsize=7)
ax.set_ylabel("fraction of signal pe"); ax.legend(fontsize=7)
fig.tight_layout()
plt.show()
print("Fraction of detected signal on each pixel type (B, G, R), by position:")
for (x0, y0), f in zip(path, fractions):
    print(f"  ({x0:g}, {y0:g}): " + ", ".join(f"{v:.3f}" for v in f))